#  NHS HES Primary Diagnosis (4C)

- Source A: NHS HES yearly workbooks (raw counts by ICD-10 4-char).
- Source B: WHO ICD-10 2019 XML (hierarchy + names).

- Year stores fiscal start year ( 2022 = Apr 2022 to Mar 2023).

# Initialization

In [20]:

import pandas as pd
pd.plotting.register_matplotlib_converters()

import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

print("Setup Complete")

Setup Complete


## File exploration

- Group A (1998-99 to 2011-12): year folders with .xls files.
- Group B (2012-13 to 2023-24): root .xlsx files with multiple sheets.

In [ ]:

# https://pbpythoncom/excel-file-combinehtml
# https://githubcom/Traceytyh/Excelpython

import os
import re
import glob

RAW_DIR = os.path.join("..", "data", "raw")

# Group A:  -xls files (1998-2011)
YEAR_FOLDERS= sorted([d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d)) and d.isdigit()])

ROOT_XLSX= sorted(glob.glob(os.path.join(RAW_DIR, "hosp-epis-stat-admi-diag-*-tab*.xlsx")))


def classify_file(filename):
    fname= filename.lower()
    if "3c" in fname:      
        return "3c"
    elif "4c" in fname:                        
        return "4c"
    else:
        return 


def extract_year_pair(filename):
    """Extract 2-digit year pair"""
 # Try 4-digit start year first (eg 2012-13)
    match = re.search(r'(\d{4})-(\d{2})', filename)         #re.search(r'(\d{4})-(\d{2})') -extract fiscal-year part of NHS file
    if match:
        return f"{match.group(1)[2:]}_{match.group(2)}"
    match = re.search(r'(\d{2})-(\d{2})', filename)
    if match:
        return f"{match.group(1)}_{match.group(2)}"
    return None

def get_primary_sheet(sheet_names, dtype):
    type_keywords = {"sum": "summary", "3c": "3 char", "4c": "4 char"}
    keyword=type_keywords[dtype]
    for name in sheet_names:
        lower= name.lower()
        if "primary" in lower and keyword in lower:
            return name
    return None

In [22]:

print(f"Group A — {len(YEAR_FOLDERS)} year folders: {YEAR_FOLDERS}")
print(f"Group B — {len(ROOT_XLSX)} root xlsx files")

Group A — 14 year folders: ['1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011']
Group B — 12 root xlsx files


# Loading the dataset

- Raw granularities: `sum`, `3c`, `4c`.
- Main target: `4c`.
- `3c` is loaded for P8 cross-checks.

### Summary DataFrames (not usingg in final output)

In [23]:

'''
# Group A — year folders (1998-2011)
for folder in YEAR_FOLDERS:
    folder_path = os.path.join(RAW_DIR, folder)
    files = [f for f in os.listdir(folder_path) if f.endswith(('.xls', '.xlsx'))]
    for f in files:
        if classify_file(f) == "sum":
            year_pair = extract_year_pair(f)
            var_name = f"df_{year_pair}_sum"
            globals()[var_name] = pd.read_excel(os.path.join(folder_path, f), sheet_name=0)
            print(f"  {var_name}: {globals()[var_name].shape}")
            break

# Group B — root xlsx workbooks (2012-2024)
for filepath in ROOT_XLSX:
    filename = os.path.basename(filepath)
    year_pair = extract_year_pair(filename)
    xl = pd.ExcelFile(filepath)
    sheet = get_primary_sheet(xl.sheet_names, "sum")
    if sheet:
        var_name = f"df_{year_pair}_sum"
        globals()[var_name] = pd.read_excel(xl, sheet_name=sheet)
        print(f"  {var_name}: {globals()[var_name].shape}")
    xl.close()

print("All loaded")
'''

'\n# Group A — year folders (1998-2011)\nfor folder in YEAR_FOLDERS:\n    folder_path = os.path.join(RAW_DIR, folder)\n    files = [f for f in os.listdir(folder_path) if f.endswith((\'.xls\', \'.xlsx\'))]\n    for f in files:\n        if classify_file(f) == "sum":\n            year_pair = extract_year_pair(f)\n            var_name = f"df_{year_pair}_sum"\n            globals()[var_name] = pd.read_excel(os.path.join(folder_path, f), sheet_name=0)\n            print(f"  {var_name}: {globals()[var_name].shape}")\n            break\n\n# Group B — root xlsx workbooks (2012-2024)\nfor filepath in ROOT_XLSX:\n    filename = os.path.basename(filepath)\n    year_pair = extract_year_pair(filename)\n    xl = pd.ExcelFile(filepath)\n    sheet = get_primary_sheet(xl.sheet_names, "sum")\n    if sheet:\n        var_name = f"df_{year_pair}_sum"\n        globals()[var_name] = pd.read_excel(xl, sheet_name=sheet)\n        print(f"  {var_name}: {globals()[var_name].shape}")\n    xl.close()\n\nprint("All l

### 3C DataFrames (kept for P8 validation)

In [24]:

# Group A: year folders(1998-2011)
for folder in YEAR_FOLDERS:
    folder_path = os.path.join(RAW_DIR, folder)
    files = [f for f in os.listdir(folder_path) if f.endswith(('.xls', '.xlsx'))]
    for f in files:
        if classify_file(f) == "3c":
            year_pair= extract_year_pair(f)
            var_name = f"df_{year_pair}_3c"
            globals()[var_name] = pd.read_excel(os.path.join(folder_path, f), sheet_name=0)
            print(f"{var_name}: {globals()[var_name].shape}")
            break

# Group B:root xlsx workbooks (2012-2024)
for filepath in ROOT_XLSX:
    filename= os.path.basename(filepath)
    year_pair = extract_year_pair(filename)
    xl = pd.ExcelFile(filepath)
    sheet =get_primary_sheet(xl.sheet_names, "3c")
    if sheet:
        var_name = f"df_{year_pair}_3c"
        globals()[var_name] = pd.read_excel(xl, sheet_name=sheet)
        print(f"{var_name}: {globals()[var_name].shape}")
    xl.close()

print("\n All loaded")

df_98_99_3c: (1615, 17)
df_99_00_3c: (1617, 17)
df_00_01_3c: (1625, 17)
df_01_02_3c: (1618, 17)
df_02_03_3c: (1614, 17)
df_03_04_3c: (1927, 17)
df_04_05_3c: (1612, 17)
df_05_06_3c: (1613, 17)
df_06_07_3c: (1609, 17)
df_07_08_3c: (1626, 18)
df_08_09_3c: (1632, 17)
df_09_10_3c: (1602, 17)
df_10_11_3c: (1600, 17)
df_11_12_3c: (1596, 17)
df_12_13_3c: (1655, 42)
df_13_14_3c: (1648, 45)
df_14_15_3c: (1643, 50)
df_15_16_3c: (1644, 50)
df_16_17_3c: (1637, 50)
df_17_18_3c: (1640, 50)
df_18_19_3c: (1638, 50)
df_19_20_3c: (1627, 50)
df_20_21_3c: (1615, 50)
df_21_22_3c: (1627, 50)
df_22_23_3c: (1623, 50)
df_23_24_3c: (1633, 50)

 All loaded


## Load 4C DataFrames

- No consistent header/sheet format across years.
- per-year maps: `HEADER_OFFSETS_4C`, `GROUP_A_SHEET_4C`.

In [ ]:
# Load all 4-Character DataFrames 

# df_yy_yy_4c

#   pdread_excel(header=None, dtype=object) https://pandaspydataorg/docs/reference/api/pandasread_excelhtml
# [pandas I/O guide for ragged Excel files](https://pandaspydataorg/docs/user_guide/iohtml#io-excel)

HEADER_OFFSETS_4C= { # someone needs to enforce NHS data publishing standards hello? took me hours T.T
 # Group A
    '98_99': 5,
    '99_00': 5,
    '00_01': 3,
    '01_02': 3,
    '02_03': 3,
    '03_04': 3,
    '04_05': 3,
    '05_06': 10,
    '06_07': 10,
    '07_08': 15,
    '08_09': 15,
    '09_10': 15,
    '10_11': 15,
    '11_12': 15,
 # Group B 
    '12_13': 17,
    '13_14': 16,
    '14_15': 10,
    '15_16': 10,
    '16_17': 10,
    '17_18': 10,
    '18_19': 10,
    '19_20': 10,
    '20_21': 10,
    '21_22': 10,
    '22_23': 10,
    '23_24': 10,
}


In [ ]:
GROUP_A_SHEET_4C= {
    '98_99': 'DIAG4DGX',
    '99_00': 'Primary Diagnosis - 4 character',
    '00_01': 'Pri Diag - 4 char',
    '01_02': 'Primary diagnosis - 4 character',
    '02_03': 'Primary Diagnosis - 4 character',
    '03_04': 'Primary Diagnosis - 4 character',
    '04_05': 'Primary Diagnosis - 4-char',
    '05_06': 'Primary diagnosis - 4 character',
    '06_07': 'Primary diagnosis - 4 character',
    '07_08': 'Primary diagnosis - 4 character',
    '08_09': 'Primary diagnosis - 4 char',
    '09_10': 'Diag4',
    '10_11': 'Diag4',
    '11_12': 'Diag4',
}


In [ ]:
def _load_4c(path, sheet, hdr):
    raw= pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    header_row = raw.iloc[hdr].tolist()
    cols= [
        (str(c).strip() if pd.notna(c) and str(c).strip() else f"_col{i}")
        for i, c in enumerate(header_row)
    ]
    df= raw.iloc[hdr + 1:].copy()
    df.columns = cols
    df= df.dropna(how='all').reset_index(drop=True)
    return df
ALL_4C = []

In [ ]:
# Group A
for folder in YEAR_FOLDERS:
    folder_path = os.path.join(RAW_DIR, folder)
    matches= [f for f in os.listdir(folder_path) if f.lower().endswith('.xls') and classify_file(f)=='4c']

    fname= matches[0]
    yp = extract_year_pair(fname)
    hdr = HEADER_OFFSETS_4C[yp]
    sheet= GROUP_A_SHEET_4C[yp]
    var= f"df_{yp}_4c"
    globals()[var]= _load_4c(os.path.join(folder_path, fname), sheet, hdr)
    ALL_4C.append(var)
    print(f"  {var}: {globals()[var].shape}")

  df_98_99_4c: (8264, 18)
  df_99_00_4c: (8263, 17)
  df_00_01_4c: (8273, 18)
  df_01_02_4c: (8240, 18)
  df_02_03_4c: (8224, 17)
  df_03_04_4c: (8214, 17)
  df_04_05_4c: (8202, 17)
  df_05_06_4c: (8149, 17)
  df_06_07_4c: (8187, 17)
  df_07_08_4c: (8126, 18)
  df_08_09_4c: (8162, 17)
  df_09_10_4c: (8060, 17)
  df_10_11_4c: (8020, 17)
  df_11_12_4c: (7988, 17)


In [ ]:
# Group B
for filepath in ROOT_XLSX:
    filename= os.path.basename(filepath)
    yp= extract_year_pair(filename)
    hdr= HEADER_OFFSETS_4C.get(yp)
    if hdr is None:
        print(f"no header offset for {yp} ({filename})")
        continue
    xl =pd.ExcelFile(filepath)
    sheet= get_primary_sheet(xl.sheet_names, '4c')
    xl.close()
    if sheet is None:
        print(f"no 4C sheet found in {filename}")
        continue
    var= f"df_{yp}_4c"
    globals()[var] = _load_4c(filepath, sheet, hdr)
    ALL_4C.append(var)
    print(f"  {var}: {globals()[var].shape}")


  df_12_13_4c: (8241, 43)
  df_13_14_4c: (8163, 45)
  df_14_15_4c: (8157, 50)
  df_15_16_4c: (8226, 50)
  df_16_17_4c: (8173, 50)
  df_17_18_4c: (8158, 50)
  df_18_19_4c: (8123, 50)
  df_19_20_4c: (8088, 50)
  df_20_21_4c: (7957, 50)
  df_21_22_4c: (8030, 50)
  df_22_23_4c: (8067, 50)
  df_23_24_4c: (8092, 50)


In [30]:
print(f"\n Loaded {len(ALL_4C)} 4C Dfs")


 Loaded 26 4C Dfs


In [31]:
#df_99_98_4c

## P2: Data cleanup

- Normalize each yearly frame
-  reliable 4-char code extraction + metric columns only.

In [ ]:
# all 4C DataFrames
# Strip footnote symbols
# Extract ICD_Code_4Char

CODE4_RE= re.compile(r'^([A-Z]\d{2}\.[0-9X])', re.I)
TOTAL_RE= re.compile(r'^\s*total\b', re.I)
_FOOTNOTE_MARKS ='‡†§* '  # ‡ † § * NBSP
_STRIP_TABLE = str.maketrans('', '', _FOOTNOTE_MARKS)
def _clean_text(v):
    if pd.isna(v):
        return ''
    return str(v).translate(_STRIP_TABLE).strip()


COL_RENAME_4C = {
    'finished consultant episodes': 'Finished Consultant Episodes (FCE)',
    'finished consultant episode':  'Finished Consultant Episodes (FCE)',
    'fce':                          'Finished Consultant Episodes (FCE)',
    'admissions':                   'Finished Admission Episodes (FAE)',
    'finished admission episodes':  'Finished Admission Episodes (FAE)',
    'fae':                          'Finished Admission Episodes (FAE)',
    'emergency':                    'Emergency',
    'waiting list':                 'Waiting List',
    'mean waiting time':            'Mean Waiting Time',
    'mean time waited':             'Mean Waiting Time',
    'median waiting time':          'Median Waiting Time',
    'median time waited':           'Median Waiting Time',
    'mean length of stay':          'Mean Length of Stay',
    'median length of stay':        'Median Length of Stay',
    'mean age':                     'Mean Age',
    'day case':                     'Day Case',
    'bed days (fce)':               'Bed Days (FCE)',
    'fce bed days':                 'Bed Days (FCE)',
    'bed days':                     'Bed Days (FCE)',
}




# keywords for headers to drop
DROP_TOKENS_4C={
    'male','female','gender unknown',
    'planned','other admission method','other','elective',
    'bed days (fae)','fae bed days',
    'day case (fae)', 'emergency (fae)',
}

AGE_BAND_RE= re.compile(r'^age\s+\d', re.I) # :) never liked regex


def _norm(c):
    s= str(c).strip().lower().replace('\n',' ')
    s= re.sub(r'\s*\((days|years)\)\s*',' ',s)
   
    if 'bed days' not in s:
        s= re.sub(r'\s*\((fce|fae)\)\s*',' ',s)
    s= re.sub(r'\s+', ' ', s).strip()
    return s

#MAIN CLEAN
def clean_4c(df):
    df= df.copy()

 #Extract ICD_Code_4Char
    codes= []
    for i in range(len(df)):
        v0= _clean_text(df.iloc[i,0])
        v1= _clean_text(df.iloc[i,1]) if df.shape[1] > 1 else ''
        m =CODE4_RE.match(v0) or CODE4_RE.match(v1)
        if m:
            codes.append(m.group(1).upper())
        elif TOTAL_RE.match(v0) or TOTAL_RE.match(v1) or i==0:
            codes.append('000')
        else:
            codes.append(None)
    df.insert(0, 'ICD_Code_4Char', codes)
    df = df[df['ICD_Code_4Char'].notna()].reset_index(drop=True)

 #Drop null col
    df=df.dropna(axis=1,how='all')

    to_drop= []
    for c in list(df.columns)[1:3]:
        body=df[c].dropna().astype(str).head(30)
        text_ct= body.str.contains(r'[A-Za-z]', regex=True).sum()
        num_ct= body.str.match(r'^-?\d+(\.\d+)?$').sum()
        if text_ct > num_ct:
            to_drop.append(c)
    df=df.drop(columns=to_drop)

    keep= ['ICD_Code_4Char']
    renames={}
    seen=set()
    for c in df.columns:
        if c=='ICD_Code_4Char':
            continue
        n = _norm(c)
        if not n or n=='nan' or n.startswith('_col'):
            continue
        if n in DROP_TOKENS_4C:
            continue
        if n in COL_RENAME_4C:
            tgt = COL_RENAME_4C[n]
            if tgt in seen:
                continue
            renames[c] = tgt
            seen.add(tgt)
            keep.append(c)
            continue
        if AGE_BAND_RE.match(n):
            pretty= 'Age '+n.split(' ',1)[1]
            if pretty in seen:
                continue
            renames[c] = pretty
            seen.add(pretty)
            keep.append(c)
            continue
    df= df[keep].rename(columns=renames)
    df=df.loc[:,~df.columns.duplicated()]
    return df


In [33]:
# For al 26 4C df
for name in ALL_4C:
    globals()[name]= clean_4c(globals()[name])
    d= globals()[name]
    print(f"{name}: {d.shape[0]:>5} rows, {d.shape[1]:>2} cols")

df_98_99_4c:  8264 rows, 16 cols
df_99_00_4c:  8263 rows, 16 cols
df_00_01_4c:  8273 rows, 16 cols
df_01_02_4c:  8240 rows, 16 cols
df_02_03_4c:  8224 rows, 16 cols
df_03_04_4c:  8214 rows, 16 cols
df_04_05_4c:  8202 rows, 16 cols
df_05_06_4c:  8149 rows, 16 cols
df_06_07_4c:  8187 rows, 16 cols
df_07_08_4c:  8125 rows, 16 cols
df_08_09_4c:  8155 rows, 16 cols
df_09_10_4c:  8053 rows, 16 cols
df_10_11_4c:  8014 rows, 16 cols
df_11_12_4c:  7981 rows, 16 cols
df_12_13_4c:  8234 rows, 36 cols
df_13_14_4c:  8155 rows, 36 cols
df_14_15_4c:  8149 rows, 36 cols
df_15_16_4c:  8220 rows, 36 cols
df_16_17_4c:  8167 rows, 36 cols
df_17_18_4c:  8150 rows, 36 cols
df_18_19_4c:  8115 rows, 36 cols
df_19_20_4c:  8078 rows, 36 cols
df_20_21_4c:  7949 rows, 36 cols
df_21_22_4c:  8023 rows, 36 cols
df_22_23_4c:  8063 rows, 36 cols
df_23_24_4c:  8088 rows, 36 cols


## Type casting and Max Age Group


- `idxmax`.



In [ ]:
# pandas "Nullable integer data type" (Int64):https://pandaspydataorg/docs/user_guide/integer_nahtml
# pdto_numeric(errors='coerce') — converts unparseable tokens to NaN: https://pandaspydataorg/docs/reference/api/pandasto_numerichtml
# DataFramesum(min_count=1) — propagate NA when every cell in a row: is NA (instead of the default treat-NA-as-0): https://pandaspydataorg/docs/reference/api/pandasDataFramesumhtml
# DataFrameidxmax — returns the index of the first maximum along axis: https://pandaspydataorg/docs/reference/api/pandasDataFrameidxmaxhtml


import numpy as np

PLACEHOLDERS_4C= ['*','-','.','..',':', '','nan', 'NaN', 'NA','N/A']
FLOAT_KEYWORDS= ('mean','median')

# Canonical 4 age buckets used for Max Age Group
AGE_BUCKETS_GA =['Age 0-14','Age 15-59','Age 60-74','Age 75+']
AGE_BUCKETS_GB= {
    'Age 0-14':  ['Age 0', 'Age 1-4', 'Age 5-9', 'Age 10-14'],
    'Age 15-59': ['Age 15','Age 16','Age 17','Age 18','Age 19','Age 20-24', 'Age 25-29', 'Age 30-34', 'Age 35-39',
'Age 40-44','Age 45-49', 'Age 50-54', 'Age 55-59'],
    'Age 60-74': ['Age 60-64', 'Age 65-69', 'Age 70-74'],
    'Age 75+':   ['Age 75-79','Age 80-84','Age 85-89','Age 90+'],
}



In [35]:

def cast_4c(df,id_cols=("ICD_Code_4Char",),placeholders=PLACEHOLDERS_4C,float_keywords=FLOAT_KEYWORDS):
    df= df.copy()

    for c in id_cols:
        if c in df.columns:
            df[c]= df[c].astype( 'string')

    for c in df.columns:
        if c in id_cols:
            continue
        s= df[c] 
        if s.dtype ==object:
            s= s.replace(placeholders, np.nan)
        s= pd.to_numeric(s,errors='coerce')
        if any(kw in str(c).lower() for kw in float_keywords): #one of these days, ill stop using 1 letter variables, i promise
            s= s.astype('float64')
        else:
            non_null= s.dropna()
            if len(non_null) and (non_null % 1 == 0).all():
                s= s.astype('Int64')
            else:
                s= s.astype('float64')
        df[c]=s
# Max Age Group via argmax s
    cols = set(df.columns)
    if set(AGE_BUCKETS_GA).issubset(cols):
        sums = df[AGE_BUCKETS_GA].copy()
    else:
        sums = pd.DataFrame(index=df.index)
        for bucket, fine in AGE_BUCKETS_GB.items():
            present= [c for c in fine if c in cols]
            sums[bucket]= df[present].sum(axis=1, min_count=1)
    sums= sums.fillna(0)
    row_sum= sums.sum(axis=1)
    df['Max Age Group']= sums.idxmax(axis=1).where(row_sum >0, pd.NA).astype('string')
    return df
for name in ALL_4C:
    globals()[name] = cast_4c(globals()[name])
    d = globals()[name]
    dtypes = dict(d.dtypes.astype(str).value_counts())
    mag_nn = d['Max Age Group'].notna().sum()

## Reshape and concatenate

- Derive Year from df_YY_YY_4c name (fiscal start year).


In [36]:

# RangeIndex https://pandaspydataorg/docs/reference/api/pandasconcathtml
YEAR_RE_4C = re.compile(r'df_(\d{2})_\d{2}_4c')

def _extract_year_4c(name):
    yy= int(YEAR_RE_4C.match(name).group(1))
    return (1900 + yy if yy >= 90 else 2000 + yy)


CANON_AGE_4C= ['Age 0-14','Age 15-59','Age 60-74','Age 75+']

CANON_ORDER_4C = [
    'Year',
    'ICD_Code_4Char',
    'Finished Consultant Episodes (FCE)',
    'Finished Admission Episodes (FAE)',
    'Emergency',
    'Waiting List',
    'Mean Waiting Time',
    'Median Waiting Time',
    'Mean Length of Stay',
    'Median Length of Stay',
    'Mean Age',
    'Age 0-14',
    'Age 15-59',
    'Age 60-74',
    'Age 75+',
    'Day Case',
    'Bed Days (FCE)',
    'Max Age Group',
]


In [37]:

def reshape_4c(df, name):
    df = df.copy()
 #1 Bucket GropuB >= 4 age bands
    present_canon = set(CANON_AGE_4C) & set(df.columns)
    if len(present_canon)==4:
        pass  #Group A: already 4
    elif len(present_canon)==0:
        for bucket, fine in AGE_BUCKETS_GB.items():
            present= [c for c in fine if c in df.columns]
            df[bucket] = df[present].sum(axis=1, min_count=1).astype('Int64')
            df[bucket]= df[present].sum(axis=1,min_count=1).astype('Int64')
        to_drop= [c for fine in AGE_BUCKETS_GB.values()
                     for c in fine if c in df.columns]
        df= df.drop(columns=to_drop)
    else:
        raise ValueError(
            f"mistakes were made"
        )
    
 # 2 Year enrich
    year = _extract_year_4c(name)
    if 'Year' in df.columns:
        df=df.drop(columns='Year')
    
    df.insert(0, 'Year', pd.Series([year] * len(df), dtype='Int64'))
 # 3 Reorder
    df=df[[c for c in CANON_ORDER_4C if c in df.columns]]
    return df



In [38]:
#df

In [ ]:

for name in ALL_4C:
    globals()[name] = reshape_4c(globals()[name], name)

schemas = {tuple(globals()[n].columns) for n in ALL_4C}
if len(schemas) != 1:
 # mismatched schema groups checking
    by_schema = {}
    for n in ALL_4C:
        by_schema.setdefault(tuple(globals()[n].columns), []).append(n)
    groups = list(by_schema.items())
    ref_cols, ref_names = groups[0]
    diffs = []
    for cols, names in groups[1:]:
        missing = set(ref_cols) - set(cols)
        extra = set(cols) - set(ref_cols)
        diffs.append(
            f" group of {len(names)} ({names[:3]}) vs group of {len(ref_names)} "
            f"({ref_names[:3]}…):\n missing: {sorted(missing)}\n extras: {sorted(extra)}"
        )
    raise AssertionError(
        f"hello darkness my old friend"
    )

df_4c_all = pd.concat([globals()[n] for n in ALL_4C], ignore_index=True)
print(f"df_4c_all shape: {df_4c_all.shape}")
print(f"Years: {sorted(df_4c_all['Year'].unique().tolist())}")
print(f"\nRows per Year:")
print(df_4c_all['Year'].value_counts().sort_index().to_string())
print(f"\nDtypes:")
print(df_4c_all.dtypes.to_string())

df_4c_all shape: (211735, 18)
Years: [1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

Rows per Year:
Year
1998    8264
1999    8263
2000    8273
2001    8240
2002    8224
2003    8214
2004    8202
2005    8149
2006    8187
2007    8125
2008    8155
2009    8053
2010    8014
2011    7981
2012    8234
2013    8155
2014    8149
2015    8220
2016    8167
2017    8150
2018    8115
2019    8078
2020    7949
2021    8023
2022    8063
2023    8088

Dtypes:
Year                                    Int64
ICD_Code_4Char                         string
Finished Consultant Episodes (FCE)      Int64
Finished Admission Episodes (FAE)       Int64
Emergency                               Int64
Waiting List                            Int64
Mean Waiting Time                     float64
Median Waiting Time                   float64
Mean Length of Stay                   float64
Median Length of Stay   

In [40]:
df_98_99_4c

,Year,ICD_Code_4Char,Finished Consultant Episodes (FCE),Finished Admission Episodes (FAE),Emergency,Waiting List,Mean Waiting Time,Median Waiting Time,Mean Length of Stay,Median Length of Stay,Mean Age,Age 0-14,Age 15-59,Age 60-74,Age 75+,Day Case,Bed Days (FCE),Max Age Group
0,1998,000,11983893,11016652,4596139,4623226,98.8,45.0,8.4,2.0,45.0,1703763,5647613,2388023,2220820,3420795,5.029468e+07,Age 15-59
1,1998,A00.1,2,1,0,2,NaN,NaN,NaN,NaN,32.0,0,2,0,0,2,NaN,Age 15-59
2,1998,A00.9,6,5,6,0,NaN,NaN,4.8,4.0,40.0,1,3,2,0,0,2.817912e+01,Age 15-59
3,1998,A01.0,116,103,112,1,NaN,NaN,7.5,7.0,21.0,41,73,2,0,0,7.699611e+02,Age 15-59
4,1998,A01.1,44,39,42,0,NaN,NaN,6.9,6.0,28.0,3,41,0,0,0,2.835713e+02,Age 15-59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8259,1998,Z99.1,54,12,28,18,1.0,1.0,30.3,6.0,44.0,3,21,13,17,0,2.325189e+02,Age 15-59
8260,1998,Z99.2,8,5,3,3,15.0,15.0,3.2,4.0,48.0,0,5,3,0,0,2.107537e+01,Age 15-59
8261,1998,Z99.3,5,4,2,1,33.0,33.0,29.4,6.0,69.0,0,1,1,3,0,8.437315e+01,Age 75+
8262,1998,Z99.8,3,3,1,1,7.0,7.0,5.0,5.0,73.0,0,1,0,2,0,1.510274e+01,Age 75+


In [ ]:
# Trim
METRIC_COLS_4C = [
    'Finished Consultant Episodes (FCE)','Finished Admission Episodes (FAE)',
    'Emergency','Waiting List',
    'Mean Waiting Time','Median Waiting Time',
    'Mean Length of Stay','Median Length of Stay',
    'Mean Age',
    'Age 0-14','Age 15-59','Age 60-74','Age 75+',
    'Day Case','Bed Days (FCE)',
]

before = len(df_4c_all)
all_na_mask = df_4c_all[METRIC_COLS_4C].isna().all(axis=1)
dropped = df_4c_all[all_na_mask]
print(f"Dropping {all_na_mask.sum()} rows with all metrics NaN")
if all_na_mask.sum():
    print(f" Years: {sorted(dropped['Year'].unique().tolist())}")
    print(f" Codes: {sorted(dropped['ICD_Code_4Char'].unique().tolist())[:20]}")

df_4c_all = df_4c_all[~all_na_mask].reset_index(drop=True)
print(f"\ndf_4c_all: {before} -> {len(df_4c_all)} rows "
      f"({before - len(df_4c_all)} dropped)")
print(f"Columns ({df_4c_all.shape[1]}): {df_4c_all.columns.tolist()}")

Dropping 13 rows with all metrics NaN
  Years: [2021]
  Codes: ['O04.2', 'O04.7', 'O05.1', 'O05.3', 'O05.4', 'O05.6', 'O05.9', 'O06.3', 'O06.6', 'O06.8', 'O07.6', 'O08.4', 'O08.6']

df_4c_all: 211735 -> 211722 rows (13 dropped)
Columns (18): ['Year', 'ICD_Code_4Char', 'Finished Consultant Episodes (FCE)', 'Finished Admission Episodes (FAE)', 'Emergency', 'Waiting List', 'Mean Waiting Time', 'Median Waiting Time', 'Mean Length of Stay', 'Median Length of Stay', 'Mean Age', 'Age 0-14', 'Age 15-59', 'Age 60-74', 'Age 75+', 'Day Case', 'Bed Days (FCE)', 'Max Age Group']


In [44]:
#df_4c_all
#looks good for now :))

## Enrichment from WHO ICD-10 XML

In [ ]:

import xml.etree.ElementTree as ET
import pickle
from collections import defaultdict
#here goes nothing
ICD_XML= os.path.join('..', 'data', 'raw', 'ICD Hierarchy Data', 'icd102019en.xml')
LOOKUPS_PKL= os.path.join('..', 'data', 'processed', 'icd10_lookups.pkl')
def _preferred_label(cls):
    #Return the text of <Rubric kind='preferred'><Label> or None
    #Per ClaML spec, each <Class> may carry multiple <Rubric> children (preferred, inclusion, exclusion, etc). We want the 'preferred' one
    for rub in cls.findall('Rubric'):
        if rub.attrib.get('kind')=='preferred':
            lbl= rub.find('Label')
            if lbl is not None and lbl.text:
                return lbl.text.strip()
    return None

def _superclass(cls):
    #Return the @code of this class's <SuperClass> parent, or None
    sc= cls.find('SuperClass')
    return sc.attrib['code'] if sc is not None else None


def build_icd10_lookups(xml_path):
    #4 dicts (chapters,blocks,cat3,cat4)
    tree= ET.parse(xml_path)
    root= tree.getroot()
    cls_kind, cls_super, cls_name= {},{},{}
    for c in root:
        if c.tag!='Class':
            continue
        code=c.attrib['code']
        cls_kind[code]=c.attrib.get('kind')
        cls_super[code]= _superclass(c)
        cls_name[code]= _preferred_label(c)

    def ancestor_chapter(code):
        cur = code
        while cur is not None:
            if cls_kind.get(cur) == 'chapter':
                return cur
            cur = cls_super.get(cur)
        return None

    def ancestor_block(code):
        cur= cls_super.get(code)
        while cur is not None:
            if cls_kind.get(cur)=='block':
                return cur
            cur = cls_super.get(cur)
        return None

    chapters, blocks, cat3, cat4 = {},{},{},{}
    for code, kind in cls_kind.items():
        if kind=='chapter':
            chapters[code]= {'code': code,'name': cls_name[code]}
        elif kind=='block':
            blocks[code]= {'code': code, 'name': cls_name[code],
                            'chapter': ancestor_chapter(code)}
        elif kind=='category':
            if len(code)==3:
                cat3[code]= {'code': code,'name': cls_name[code],'group': ancestor_block(code),'chapter': ancestor_chapter(code)}
            elif len(code)==5:
                cat4[code] = {'code': code,'name': cls_name[code],'code_3c': code[:3]}

    for c4, info in cat4.items():
        c3= info['code_3c']
        if c3 in cat3:
            info['chapter']= cat3[c3]['chapter']
            info['group']= cat3[c3]['group']
        else:
            info['chapter']=None
            info['group']=None

    ch_codes= defaultdict(list)
    for c3,info in cat3.items():
        ch_codes[info['chapter']].append(c3)
    for ch,codes in ch_codes.items():
        codes.sort()
        if ch in chapters:
            chapters[ch]['code_range'] = f"{codes[0]}-{codes[-1]}"

    if 'U07.0' in cat4 and not cat4['U07.0'].get('name'):
        cat4['U07.0']['name'] = 'Vaping-related disorder'

    return {'chapters': chapters, 'blocks': blocks,
            'cat3': cat3, 'cat4': cat4}

LOOKUPS = build_icd10_lookups(ICD_XML)
os.makedirs(os.path.dirname(LOOKUPS_PKL), exist_ok=True)
with open(LOOKUPS_PKL, 'wb') as f:
    pickle.dump(LOOKUPS, f)

print(f"XML: {len(LOOKUPS['chapters'])} chapters, "
      f"{len(LOOKUPS['blocks'])} blocks, "
      f"{len(LOOKUPS['cat3'])} 3-char, "
      f"{len(LOOKUPS['cat4'])} 4-char")
print(f"Persisted to {LOOKUPS_PKL}")
CHAPTERS= LOOKUPS['chapters']
BLOCKS= LOOKUPS['blocks']
CAT3= LOOKUPS['cat3']
CAT4= LOOKUPS['cat4']

def derive_code_3c(code4):
    #ICD-10 parent: first 3 chars of a 4-char code ('A00.0'->'A00', 'A09.X'->'A09')
    if pd.isna(code4) or code4 == '000':
        return pd.NA
    return code4[:3]

df_4c_all['ICD_Code_3_Char'] = df_4c_all['ICD_Code_4Char'].map(derive_code_3c).astype('string')

def enrich_row(code4, code3):
    #Return (chapter, chapter_name, group, group_name, name_3c, name_4c)
    if pd.isna(code4) or code4 == '000':
        return (pd.NA, pd.NA, pd.NA, pd.NA, pd.NA, pd.NA)
    c3_info = CAT3.get(code3)
    if c3_info is None:
        return (pd.NA, pd.NA, pd.NA, pd.NA, pd.NA, pd.NA)
    chap = c3_info['chapter']
    grp = c3_info['group']
    chap_name = CHAPTERS.get(chap, {}).get('name') if chap else None
    grp_name = BLOCKS.get(grp, {}).get('name') if grp else None
    name_3c = c3_info['name']
    c4_info = CAT4.get(code4)
    name_4c = c4_info['name'] if c4_info else pd.NA
    return (chap or pd.NA, chap_name or pd.NA,
            grp or pd.NA, grp_name or pd.NA,
            name_3c or pd.NA, name_4c) #why am i doing this even lmao.

enriched = df_4c_all.apply(
    lambda r: enrich_row(r['ICD_Code_4Char'],r['ICD_Code_3_Char']),
    axis=1, result_type='expand'
)
enriched.columns = ['Chapter','Chapter_Name','Group','Group_Name','Code_Name_3C', 'ICD_Code_Name_4C']
for c in enriched.columns:
    df_4c_all[c] = enriched[c].astype('string')

# unmapped codes check
non_total= df_4c_all[df_4c_all['ICD_Code_4Char'] != '000']
unmapped_chapter= non_total[non_total['Chapter'].isna()]
unmapped_4c_name= non_total[non_total['ICD_Code_Name_4C'].isna()]
if len(unmapped_chapter):
    print("\nCodes with NO chapter mapping:")
    print(unmapped_chapter['ICD_Code_4Char'].value_counts().head(20).to_string())

print(f"\ndf_4c_all now: {df_4c_all.shape}")
print(f"Columns: {df_4c_all.columns.tolist()}")

Parsed XML: 22 chapters, 274 blocks, 2090 3-char, 9153 4-char
Persisted to ./data/processed/icd10_lookups.pkl

Rows missing Chapter (non-'000'): 266 (22 distinct codes)
Rows missing ICD_Code_Name_4C (non-'000'): 15239 (717 distinct codes)

Codes with NO chapter mapping:
ICD_Code_4Char
B59.X    26
A90.X    19
I84.0    19
I84.1    19
I84.2    19
I84.4    19
I84.5    19
I84.6    19
I84.7    19
I84.8    19
I84.9    19
I84.3    18
A91.X    17
U06.0     3
U06.1     3
U80.0     2
U80.1     2
U89.8     1
U80.8     1
U06.9     1

df_4c_all now: (211722, 25)
Columns: ['Year', 'ICD_Code_4Char', 'Finished Consultant Episodes (FCE)', 'Finished Admission Episodes (FAE)', 'Emergency', 'Waiting List', 'Mean Waiting Time', 'Median Waiting Time', 'Mean Length of Stay', 'Median Length of Stay', 'Mean Age', 'Age 0-14', 'Age 15-59', 'Age 60-74', 'Age 75+', 'Day Case', 'Bed Days (FCE)', 'Max Age Group', 'ICD_Code_3_Char', 'Chapter', 'Chapter_Name', 'Group', 'Group_Name', 'Code_Name_3C', 'ICD_Code_Name_4C']


## 6.5: Final Grid + Impute

In [ ]:


import numpy as np

# 1) one row per WHO 4-char code
sk_rows = []
for c4, info in CAT4.items():
    c3 = info['code_3c']
    c3_info = CAT3.get(c3)
    if c3_info is None:
        continue  # skip orphan codes
    chap = c3_info['chapter']; grp = c3_info['group']
    sk_rows.append({
        'ICD_Code_4Char': c4,
        'ICD_Code_3_Char': c3,
        'Chapter': chap,
        'Chapter_Name': CHAPTERS.get(chap, {}).get('name') if chap else None,
        'Group': grp,
        'Group_Name': BLOCKS.get(grp, {}).get('name') if grp else None,
        'Code_Name_3C': c3_info['name'],
        'ICD_Code_Name_4C': info['name'],
    })
skeleton = pd.DataFrame(sk_rows)
for c in skeleton.columns:
    skeleton[c] = skeleton[c].astype('string')
print('WHO 4-char rows:',len(skeleton))

years= sorted(df_4c_all['Year'].dropna().unique().tolist())

yr_df= pd.DataFrame({'Year': pd.array(years,dtype='Int64')})
grid = yr_df.merge(skeleton, how='cross')
print('Cartesian grid shape:', grid.shape, '(expected', len(years)*len(skeleton), ')')

# 3 left-join
obs = df_4c_all.drop(columns=[
    'ICD_Code_3_Char','Chapter','Chapter_Name','Group','Group_Name',
    'Code_Name_3C','ICD_Code_Name_4C'
]).copy()

obs = obs[obs['ICD_Code_4Char'] != '000'].copy() # drop Total rows
# align join-key dtypes so merges right
grid['ICD_Code_4Char']= grid['ICD_Code_4Char'].astype('string')
obs['ICD_Code_4Char']  = obs['ICD_Code_4Char'].astype('string')
grid['Year']= grid['Year'].astype('Int64')
obs['Year']= obs['Year'].astype('Int64')
df_4c_dense= grid.merge(obs, on=['Year','ICD_Code_4Char'], how='left')
print('After left-join:', df_4c_dense.shape)

COUNT_FILL_COLS= [
    'Finished Consultant Episodes (FCE)','Finished Admission Episodes (FAE)',
    'Emergency', 'Waiting List',
    'Age 0-14','Age 15-59','Age 60-74','Age 75+',
    'Day Case', 'Bed Days (FCE)',
]
NULL_FILL_COLS= [
    'Mean Waiting Time','Median Waiting Time',
    'Mean Length of Stay', 'Median Length of Stay',
    'Mean Age',
]
for c in COUNT_FILL_COLS:
    if c in df_4c_dense.columns:
        df_4c_dense[c] = df_4c_dense[c].fillna(0)

# NULL_FILL_COLS not imputed

ICD_INTRODUCED = {
 # SARS
    'U04':2003,'U04.9':2003,
 # COVID-19 + vaping uwu
    'U07.0':2019,'U07.1':2019,'U07.2': 2019,
 # U06 provisional placeholders
    'U06':2019,'U06.0': 2019,'U06.9': 2019,
}

_intro_map= df_4c_dense['ICD_Code_4Char'].map(ICD_INTRODUCED)
pre_intro_mask = _intro_map.notna() & (df_4c_dense['Year'] < _intro_map)
n_pre_intro= int(pre_intro_mask.sum())
if n_pre_intro:
    for c in COUNT_FILL_COLS:
        if c in df_4c_dense.columns:
            df_4c_dense.loc[pre_intro_mask, c]= pd.NA
print(f'({len(ICD_INTRODUCED)} codes x pre-intro years)')

if 'Max Age Group' in df_4c_dense.columns:
    df_4c_dense['Max Age Group']= df_4c_dense['Max Age Group'].astype('string')

print('Final shape:', df_4c_dense.shape)
print('Nulls in hierarchy cols:')
for c in ['Chapter','Chapter_Name','Group','Group_Name','ICD_Code_3_Char','Code_Name_3C','ICD_Code_4Char','ICD_Code_Name_4C']:
    print(f' {c}:{df_4c_dense[c].isna().sum()}')
print('Zero-imputed rows:', (df_4c_dense['Finished Consultant Episodes (FCE)']==0).sum())
print('Null mean/ median counts:')
for c in NULL_FILL_COLS:
    print(f' {c}:{df_4c_dense[c].isna().sum()}')

WHO 4-char rows: 9153
Cartesian grid shape: (237978, 9) (expected 237978 )
After left-join: (237978, 25)
(8 codes x pre-intro years)
Final shape: (237978, 25)
Nulls in hierarchy cols:
 Chapter:0
 Chapter_Name:0
 Group:0
 Group_Name:0
 ICD_Code_3_Char:0
 Code_Name_3C:0
 ICD_Code_4Char:0
 ICD_Code_Name_4C:26
Zero-imputed rows: 41673
Null mean/ median counts:
 Mean Waiting Time:63020
 Median Waiting Time:63026
 Mean Length of Stay:44372
 Median Length of Stay:44379
 Mean Age:42834


In [47]:
df_4c_dense.head(3)

,Year,ICD_Code_4Char,ICD_Code_3_Char,Chapter,Chapter_Name,Group,Group_Name,Code_Name_3C,ICD_Code_Name_4C,Finished Consultant Episodes (FCE),...,Mean Length of Stay,Median Length of Stay,Mean Age,Age 0-14,Age 15-59,Age 60-74,Age 75+,Day Case,Bed Days (FCE),Max Age Group
0,1998,A00.0,A00,I,Certain infectious and parasitic diseases,A00-A09,Intestinal infectious diseases,Cholera,"Cholera due to Vibrio cholerae 01, biovar chol...",0,...,NaN,NaN,NaN,0,0,0,0,0,0.0,<NA>
1,1998,A00.1,A00,I,Certain infectious and parasitic diseases,A00-A09,Intestinal infectious diseases,Cholera,"Cholera due to Vibrio cholerae 01, biovar eltor",2,...,NaN,NaN,32.0,0,2,0,0,2,0.0,Age 15-59
2,1998,A00.9,A00,I,Certain infectious and parasitic diseases,A00-A09,Intestinal infectious diseases,Cholera,"Cholera, unspecified",6,...,4.8,4.0,40.0,1,3,2,0,0,28.179123,Age 15-59


##  Age % conversion + final CSV


In [48]:
# reorder columns
# min_count=1 preserves all-NA rows as NaN

import numpy as np

AGE_COUNT_COLS= ['Age 0-14','Age 15-59','Age 60-74','Age 75+']
AGE_PCT_COLS= ['Age 0-14 %','Age 15-59 %','Age 60-74 %','Age 75+ %']

_age_sum_s= df_4c_dense[AGE_COUNT_COLS].sum(axis=1, min_count=1).astype('float64')
_all_na= _age_sum_s.isna().to_numpy()
_age_sum= _age_sum_s.fillna(0).to_numpy()
_safe_sum= np.where(_age_sum > 0, _age_sum, 1.0)  # placeholder divisor
for cnt, pct in zip(AGE_COUNT_COLS, AGE_PCT_COLS):
    cnt_arr= df_4c_dense[cnt].astype('float64').to_numpy()
    pct_arr= np.where(_age_sum > 0, cnt_arr / _safe_sum * 100.0, 0.0)
    pct_arr= np.where(_all_na, np.nan, pct_arr)  # keep NaN where counts absent
    df_4c_dense[pct]= np.round(pct_arr, 2)

FINAL_COLS_4C = [
    'Year',
    'Chapter', 'Chapter_Name',
    'Group', 'Group_Name',
    'ICD_Code_3_Char', 'Code_Name_3C',
    'ICD_Code_4Char','ICD_Code_Name_4C',
    'Finished Consultant Episodes (FCE)', 'Finished Admission Episodes (FAE)',
    'Emergency', 'Waiting List',
    'Mean Waiting Time', 'Median Waiting Time',
    'Mean Length of Stay', 'Median Length of Stay',
    'Mean Age',
    'Age 0-14 %','Age 15-59 %','Age 60-74 %','Age 75+ %',
    'Day Case','Bed Days (FCE)','Max Age Group',
]
assert all(c in df_4c_dense.columns for c in FINAL_COLS_4C), \
    f"Missing:     {[c for c in FINAL_COLS_4C if c not in df_4c_dense.columns]}"

df_4c_final= df_4c_dense[FINAL_COLS_4C].copy()
df_4c_final = df_4c_final.sort_values(['Year','Chapter','Group','ICD_Code_3_Char','ICD_Code_4Char']).reset_index(drop=True)

OUT_PATH = '../data/processed/df_4c_final.csv'
df_4c_final.to_csv(OUT_PATH, index=False)
print('Wrote', OUT_PATH)
print('Shape:', df_4c_final.shape)
print('Columns (', len(df_4c_final.columns),'):',df_4c_final.columns.tolist())
df_4c_final.head(5)

OSError: Cannot save file into a non-existent directory: '../data/processed'

## Validation

1. Row count(years x WHO 4-char codes).
2. Column order equals TARGET_COLS.
3. Hierarchy key completeness.
4. Full code coverage per year.
5. 3C coverage against WHO.
6. 3C coverage against sibling df_3c_final.csv`.
7. Age % closure near 100 where data exists.
8. Zero-imputed row count.
9. Raw-vs-final spot check.
10. Output CSV size.
11. .X coexistence with subdivisions.
12. Max Age Group tie report.

In [53]:
TARGET_COLS = [
    'Year','Chapter','Chapter_Name','Group','Group_Name','ICD_Code_3_Char','Code_Name_3C','ICD_Code_4Char','ICD_Code_Name_4C',
    'Finished Consultant Episodes (FCE)','Finished Admission Episodes (FAE)','Emergency','Waiting List','Mean Waiting Time','Median Waiting Time',
    'Mean Length of Stay','Median Length of Stay',
    'Mean Age','Age 0-14 %','Age 15-59 %','Age 60-74 %','Age 75+ %',
    'Day Case','Bed Days (FCE)','Max Age Group',
]
n_years= df_4c_final['Year'].nunique()
n_codes= df_4c_final['ICD_Code_4Char'].nunique()
print(f'Row count: {len(df_4c_final)}== {n_years} years x {n_codes} codes = {n_years*n_codes}: 'f'{"PASS" if len(df_4c_final) == n_years*n_codes else "FAIL"}')
print(f'Column order matches target: {"PASS" if df_4c_final.columns.tolist() == TARGET_COLS else "FAIL"}')
hier_cols= ['Chapter','Chapter_Name','Group','Group_Name','ICD_Code_3_Char','Code_Name_3C','ICD_Code_4Char']
hier_nulls= {c: df_4c_final[c].isna().sum() for c in hier_cols}
print(f'Hierarchy null counts: {hier_nulls}'f'{"PASS" if all(v==0 for v in hier_nulls.values()) else "FAIL"}')
# Every WHO 4-char in every year check
per_year_codes = df_4c_final.groupby('Year')['ICD_Code_4Char'].nunique()
print(f'Codes per year all equal: min={per_year_codes.min()} max={per_year_codes.max()}: 'f'{"PASS" if per_year_codes.min() == per_year_codes.max() == n_codes else "FAIL"}')
codes_3c_in_4c = set(df_4c_final['ICD_Code_3_Char'].unique())
codes_3c_in_who = set(CAT3.keys())
missing_in_who = codes_3c_in_4c - codes_3c_in_who


Row count: 237978== 26 years x 9153 codes = 237978: PASS
Column order matches target: PASS
Hierarchy null counts: {'Chapter': np.int64(0), 'Chapter_Name': np.int64(0), 'Group': np.int64(0), 'Group_Name': np.int64(0), 'ICD_Code_3_Char': np.int64(0), 'Code_Name_3C': np.int64(0), 'ICD_Code_4Char': np.int64(0)}PASS
Codes per year all equal: min=9153 max=9153: PASS


In [54]:

# Cross-check against df_3c_final (checking manually, so dropping now, runs faster)
'''
df3 = pd.read_csv('../data/processed/df_3c_final.csv')
codes_3c_in_3cfinal = set(df3['ICD_Code'].astype(str).unique()) - {'000'}
missing_from_3c = codes_3c_in_4c - codes_3c_in_3cfinal
print(f'[6] 3C codes present in 4C but missing from df_3c_final.csv: {len(missing_from_3c)}')
if missing_from_3c:
    print(f'    sample: {sorted(missing_from_3c)[:10]}')
age_cols = ['Age 0-14 %','Age 15-59 %','Age 60-74 %','Age 75+ %']
age_sum = df_4c_final[age_cols].sum(axis=1)
has_data = df_4c_final['Finished Consultant Episodes (FCE)'] > 0
non_zero_age = has_data & (age_sum > 0)
bad = non_zero_age & ~age_sum.between(99.9, 100.1)
print(f'[7] Age % sums ≈ 100 where data present: {"PASS" if bad.sum()==0 else f"FAIL ({bad.sum()} rows)"}')
# 8) Report zero-imputed row count
zero_imp = (df_4c_final['Finished Consultant Episodes (FCE)'] == 0).sum()

sample_code= 'I21.9'; sample_yr = 2022
obs= df_22_23_4c[df_22_23_4c['ICD_Code_4Char'] == sample_code]
fin= df_4c_final[(df_4c_final['ICD_Code_4Char'] == sample_code) & (df_4c_final['Year'] == sample_yr)]
print(f'[9] Spot-check {sample_code} Year={sample_yr}:')
print(f'    raw FCE={obs["Finished Consultant Episodes (FCE)"].iloc[0] if len(obs) else "n/a"}  '
      f'final FCE={fin["Finished Consultant Episodes (FCE)"].iloc[0] if len(fin) else "n/a"}')
'''



'\ndf3 = pd.read_csv(\'../data/processed/df_3c_final.csv\')\ncodes_3c_in_3cfinal = set(df3[\'ICD_Code\'].astype(str).unique()) - {\'000\'}\nmissing_from_3c = codes_3c_in_4c - codes_3c_in_3cfinal\nprint(f\'[6] 3C codes present in 4C but missing from df_3c_final.csv: {len(missing_from_3c)}\')\nif missing_from_3c:\n    print(f\'    sample: {sorted(missing_from_3c)[:10]}\')\nage_cols = [\'Age 0-14 %\',\'Age 15-59 %\',\'Age 60-74 %\',\'Age 75+ %\']\nage_sum = df_4c_final[age_cols].sum(axis=1)\nhas_data = df_4c_final[\'Finished Consultant Episodes (FCE)\'] > 0\nnon_zero_age = has_data & (age_sum > 0)\nbad = non_zero_age & ~age_sum.between(99.9, 100.1)\nprint(f\'[7] Age % sums ≈ 100 where data present: {"PASS" if bad.sum()==0 else f"FAIL ({bad.sum()} rows)"}\')\n# 8) Report zero-imputed row count\nzero_imp = (df_4c_final[\'Finished Consultant Episodes (FCE)\'] == 0).sum()\n\nsample_code= \'I21.9\'; sample_yr = 2022\nobs= df_22_23_4c[df_22_23_4c[\'ICD_Code_4Char\'] == sample_code]\nfin= df_4c_